In [ ]:
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
import logging
from typing import List
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torch.nn as nn
import os
from utils.model import LSTNowcaster
from utils.data.TiledLandsatDataModule import TiledLandsatDataModule
from utils.voice import notifySelf, PercentageProgressCallback
# from pytorch_lightning.loggers import TensorBoardLogger
# from pytorch_lightning.profilers import PyTorchProfiler
torch.cuda.empty_cache()

os.environ["WANDB_NOTEBOOK_NAME"] = "TrainUNet-Basic.ipynb"
os.environ["WANDB_DIR"] = "./wandb"
os.environ["WANDB_CACHE_DIR"] = "./wandb/.cache/wandb"
os.environ["WANDB_CONFIG_DIR"] = "./wandb/.config/wandb"
os.environ["WANDB_DATA_DIR"] = "./wandb/.cache/wandb-data"
os.environ["WANDB_ARTIFACT_DIR"] = "./wandb/artifacts"

config = {
    "experiment_name": "Test normalization fixes",
    "debug": True,
    "by_city": False,
    "months_ahead": 0,
    "tile_size": 512,
    "tile_overlap": 0.0,
    "learning_rate": 1e-4,
    "model": "unet",
    "backbone": "resnet50",
    "dataset": "pure_landsat",
    "epochs": 350,
    "batch_size": 32,
    "pretrained_weights": True,
    "deterministic": True,
    "in_channels": 5
}

# logger = TensorBoardLogger("tb_logs", name="unet_profile")
# profiler = PyTorchProfiler(
#     on_trace_ready=torch.profiler.tensorboard_trace_handler("tb_logs/profiler0"),
#     schedule=torch.profiler.schedule(skip_first=10, wait=1, warmup=1, active=20)
# )

wandb_logger = WandbLogger(
    project="heat-island",  # your project name
    name=config['experiment_name'],  # name of this particular run
    log_model="all",  # log model checkpoints
    save_code=True,
    save_dir="./wandb",  # where to save the logs locally
)
wandb_logger.log_hyperparams(config)
if config["dataset"] == "pure_landsat":
    data_module = TiledLandsatDataModule(
        data_dir="./Data",
        monthsAhead=config["months_ahead"],
        batch_size=config["batch_size"],
        num_workers=5,
        byCity=config["by_city"],
        debug=config["debug"],
        tile_size=config["tile_size"],
        tile_overlap=config["tile_overlap"]
    )
    data_module.setup()

percentage_callback = PercentageProgressCallback(total_epochs=config["epochs"], experiment_name=config["experiment_name"])
checkpoint_callback = ModelCheckpoint(
    dirpath="./wandb/heat-island/checkpoints", monitor="val_rmse_F", mode="min", save_top_k=3, every_n_epochs=1, save_last=True
)

# Initialize trainer with explicit steps
trainer = pl.Trainer(
    max_epochs=config["epochs"],
    gradient_clip_val=0.5,
    log_every_n_steps=10,
    enable_progress_bar=True,
    enable_model_summary=False,
    deterministic=config["deterministic"],
    num_sanity_val_steps=2,
    reload_dataloaders_every_n_epochs=1,
    logger=wandb_logger,
    # profiler=profiler,
    # logger=logger,
    callbacks=[checkpoint_callback, percentage_callback]
)

In [ ]:
from utils.model import LSTNowcaster
model = LSTNowcaster(model=config["model"], backbone=config["backbone"], in_channels=config["in_channels"], learning_rate=config["learning_rate"], pretrained_weights=config["pretrained_weights"])

In [ ]:
trainer.fit(model=model, datamodule=data_module)

In [ ]:
# After training, load the best checkpoint for testing
best_model_path = checkpoint_callback.best_model_path
print(f"Best model path: {best_model_path}")

if best_model_path:
    # Load best checkpoint
    best_model = LSTNowcaster.load_from_checkpoint(
        checkpoint_path=best_model_path,
        model=config["model"],
        backbone=config["backbone"],
        in_channels=config["in_channels"],
        learning_rate=config["learning_rate"],
        pretrained_weights=config["pretrained_weights"]
    )
    
    # Test using the best model
    trainer.test(model=best_model, datamodule=data_module)
else:
    print("No checkpoint found, testing with the current model state")
    trainer.test(model=model, datamodule=data_module)

In [ ]:
# %%
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
import logging
from typing import List
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
import torch.nn as nn
import os
from utils.model import LSTNowcaster
from utils.data.TiledLandsatDataModule import TiledLandsatDataModule
from utils.voice import notifySelf
# from pytorch_lightning.loggers import TensorBoardLogger
# from pytorch_lightning.profilers import PyTorchProfiler
torch.cuda.empty_cache()
torch.set_float32_matmul_precision('high')

os.environ["WANDB_NOTEBOOK_NAME"] = "TrainUNet-Basic.ipynb"
os.environ["WANDB_DIR"] = "./wandb"
os.environ["WANDB_CACHE_DIR"] = "./wandb/.cache/wandb"
os.environ["WANDB_CONFIG_DIR"] = "./wandb/.config/wandb"
os.environ["WANDB_DATA_DIR"] = "./wandb/.cache/wandb-data"
os.environ["WANDB_ARTIFACT_DIR"] = "./wandb/artifacts"

config = {
    "experiment_name": "Test normalization fixes",
    "debug": True,
    "by_city": False,
    "months_ahead": 0,
    "tile_size": 512,
    "tile_overlap": 0.0,
    "learning_rate": 1e-4,
    "model": "unet",
    "backbone": "resnet50",
    "dataset": "pure_landsat",
    "epochs": 300,
    "batch_size": 32,
    "pretrained_weights": True,
    "deterministic": True,
    "in_channels": 5
}

for i in range(3):
    if i == 0:
        config["experiment_name"] = "Test(3) 250 Batch 32"
        config["batch_size"] = 32
    if i == 1:
        config["experiment_name"] = "Test(3) 250 Batch 4"
        config["batch_size"] = 4
    if i == 2:
        config["experiment_name"] = "Test(32) 250 Batch 1"
        config["batch_size"] = 1
        
    wandb_logger = WandbLogger(
        project="heat-island",
        name=config['experiment_name'],
        log_model="all",
        save_code=True,
        save_dir="./wandb",
    )
    wandb_logger.log_hyperparams(config)
    
    if config["dataset"] == "pure_landsat":
        data_module = TiledLandsatDataModule(
            data_dir="./Data",
            monthsAhead=config["months_ahead"],
            batch_size=config["batch_size"],
            num_workers=5,
            byCity=config["by_city"],
            debug=config["debug"],
            tile_size=config["tile_size"],
            tile_overlap=config["tile_overlap"]
        )
        data_module.setup()

    class PercentageProgressCallback(Callback):
        def __init__(self, total_epochs, experiment_name):
            super().__init__()
            self.total_epochs = total_epochs
            self.experiment_name = experiment_name

        def on_train_epoch_end(self, trainer, pl_module):
            current_epoch = trainer.current_epoch
            if current_epoch % 20 == 0:
                current_percentage = min(100, int(current_epoch / self.total_epochs * 100))
                notifySelf(f'{self.experiment_name} is at {current_percentage:.2f}%')
    percentage_callback = PercentageProgressCallback(total_epochs=config["epochs"], experiment_name=config["experiment_name"])
    checkpoint_callback = ModelCheckpoint(
        dirpath="./wandb/heat-island/checkpoints", monitor="val_rmse_F", mode="min", save_top_k=3, every_n_epochs=1, save_last=True
    )

    # Initialize trainer with explicit steps
    trainer = pl.Trainer(
        max_epochs=config["epochs"],
        gradient_clip_val=0.5,
        log_every_n_steps=10,
        enable_progress_bar=True,
        enable_model_summary=False,
        deterministic=config["deterministic"],
        num_sanity_val_steps=2,
        reload_dataloaders_every_n_epochs=1,
        logger=wandb_logger,
        callbacks=[checkpoint_callback, percentage_callback]
    )

    # Create model
    model = LSTNowcaster(
        model=config["model"], 
        backbone=config["backbone"], 
        in_channels=config["in_channels"], 
        learning_rate=config["learning_rate"], 
        pretrained_weights=config["pretrained_weights"]
    )

    # Train model
    trainer.fit(model=model, datamodule=data_module)
    
    # Clean up resources
    del model
    del trainer
    del data_module
    del wandb_logger
    del checkpoint_callback
    
    # Force garbage collection and clear CUDA cache
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    
    # Print memory stats for debugging
    if torch.cuda.is_available():
        print(f"Loop {i} completed. CUDA memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
notifySelf("Batch experiment ended.")


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jesus-guerrero (jesus-guerrero-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Preparing scene by scene...: 100%|██████████| 650/650 [00:00<00:00, 10328.02it/s]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
Preparing scene by scene...: 100%|██████████| 650/650 [00:00<00:00, 10366.98it/s]
/home/ubuntu/miniconda3/envs/ml3/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/ubuntu/heat-island-test/wandb/heat-island/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]